# TCP vs Mocap-cube sanity check

Reads the **exact current TCP** from RTDE receive (joint angles) and the **cube center** from
the mocap, maps both into the robot base / policy frame (via `base_frame_calibration.json`),
renders them together in a single MuJoCo frame, and prints the distance between the two points.

The compared TCP point is the **grasp midpoint between the two Hand-E fingers** — the sim's
`0.5*(left_finger_touch + right_finger_touch)` — with the Hand-E offset baked in exactly like
MuJoCo (computed by running the model's own FK from the measured joints). The raw RTDE flange
TCP is printed alongside for reference.

In the render: the **robot** is drawn at its measured joints, the **box** sits at the mocap
cube center, and the **red target sphere** marks the grasp midpoint. If the calibration is good
the sphere sits where the real gripper is relative to the cube (≈0 when actually grasping it).

No motion is commanded. The arm still needs the External Control program PLAYING on the
pendant, because `connect()` attaches to it (blocks until Play).

In [ ]:
# --- Imports + path setup (locate the repo root from this notebook's folder) ---
import os, sys
import numpy as np
import mujoco

here = os.getcwd()
while here != os.path.dirname(here) and not os.path.exists(
        os.path.join(here, "robots", "UR3e", "ur3_realrobot_dependencies.py")):
    here = os.path.dirname(here)
REPO_ROOT = here
UR3E_DIR = os.path.join(REPO_ROOT, "robots", "UR3e")
for p in (REPO_ROOT, UR3E_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

from ur3_realrobot_dependencies import UR3RealRobotPick
from motion_capture.mymocap.vrpn_dependencies import VRPNRigidBodyReader
print("repo root:", REPO_ROOT)

In [ ]:
# --- Config ---
ROBOT_IP = "192.168.1.4"            # UR3e PolyScope X
UR_CAP_PORT = 50002                 # External Control URCap

MOCAP_SERVER_IP = "10.1.1.198"
MOCAP_RIGID_BODY_NAME = "CubeInCube2"

MODEL_PATH = os.path.join(
    REPO_ROOT,
    "mujoco_playground/_src/manipulation/my_ur3/xmls/"
    "mjx_single_cube_position_ur3.xml",
)

In [ ]:
# --- Connect the arm (RTDE receive + External Control URCap). Press PLAY on the pendant. ---
robot = UR3RealRobotPick(host=ROBOT_IP, use_ext_urcap=True, ur_cap_port=UR_CAP_PORT)
print("Waiting for the go: press Play on the pendant (External Control PLAYING) ...")
robot.connect()
assert robot.is_connected(), "RTDE not connected — check IP + Remote Control mode."
robot.print_feedback()

# FK model — gives the Hand-E finger sites so we can compute the grasp midpoint from joints.
robot.init_fk_model(MODEL_PATH)

In [ ]:
# --- Connect the mocap (VRPN), subscribe to only the cube body ---
mocap = VRPNRigidBodyReader(
    MOCAP_SERVER_IP,
    rigid_body_name=MOCAP_RIGID_BODY_NAME,
    names=[MOCAP_RIGID_BODY_NAME],
)
assert mocap.start(timeout=5.0), "Mocap failed to subscribe — check server IP + SDK Enabled."
assert mocap.wait_for_data(timeout=5.0), (
    f"Body '{MOCAP_RIGID_BODY_NAME}' never reported — check the name + camera visibility.")
print("mocap up; raw cube (world):", np.round(mocap.get_rigid_body_xyz(), 4).tolist())

In [ ]:
# --- Read both points, map into the base/policy frame, report the distance ---
fb = robot.receive_feedback()
q = np.asarray(fb["q"], dtype=float)
tcp_flange = np.asarray(fb["tcp_xyz"], dtype=float)   # raw RTDE flange TCP (base frame) — reference only

# Grasp midpoint = 0.5*(left_finger_touch + right_finger_touch), via the model's own FK.
# This bakes in the Hand-E offset exactly like MuJoCo (sim uses this same midpoint for the
# gripper approach axis in ur3_pick). It is independent of the finger opening (the two sites
# sit symmetrically at +-0.025 on the tool axis) and of the pendant's configured TCP.
m, d = robot._fk_model, robot._fk_data
d.qpos[robot._fk_arm_qadr] = q
d.qpos[robot._fk_finger_qadr] = 0.0
mujoco.mj_forward(m, d)
l_site = d.site_xpos[robot._fk_left_touch].copy()
r_site = d.site_xpos[robot._fk_right_touch].copy()
grasp_mid = 0.5 * (l_site + r_site)                   # base frame

cube_world = mocap.get_rigid_body_xyz()
cube_quat_world = mocap.get_rigid_body_quat()
cube_base = np.asarray(robot.mocap_pos_to_base(cube_world), dtype=float)   # mocap world -> base frame
cube_quat_base = robot.mocap_quat_to_base(cube_quat_world)

dist = float(np.linalg.norm(grasp_mid - cube_base))

print(f"RTDE flange TCP (base, m): {np.round(tcp_flange, 4).tolist()}   (reference)")
print(f"grasp midpoint  (base, m): {np.round(grasp_mid, 4).tolist()}   (Hand-E offset, = sim 0.5*(L+R))")
print(f"cube center     (base, m): {np.round(cube_base, 4).tolist()}")
print(f"delta (m)               : {np.round(grasp_mid - cube_base, 4).tolist()}")
print(f"DISTANCE                : {dist * 1000:.1f} mm ({dist:.4f} m)")

In [ ]:
# --- Render one MuJoCo frame: robot at measured q, box = cube center, red sphere = grasp midpoint ---
import matplotlib.pyplot as plt

mj = robot.mujoco_init_model(
    MODEL_PATH, height=600, width=800,
    cam_lookat=(0.3, 0.0, 0.2), cam_distance=1.1, cam_azimuth=130, cam_elevation=-20,
)
# box at the cube center (with mocap orientation); target sphere (mocap 0) at the grasp midpoint.
frame = robot.mujoco_sync_and_render(
    mj, q, gripper_ctrl=0.0, box_pos=cube_base, drop_target=grasp_mid, box_quat=cube_quat_base,
)
plt.figure(figsize=(9, 7))
plt.imshow(frame)
plt.title(f"box=cube (mocap), red sphere=grasp midpoint (Hand-E)  —  distance {dist*1000:.1f} mm")
plt.axis("off")
plt.show()

In [ ]:
# --- Disconnect ---
mocap.stop()
robot.disconnect()
print("done.")